In [1]:
# Cell 1: Imports and environment setup
from pathlib import Path
import re
import sys

import gymnasium as gym
import torch
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.monitor import Monitor

# Resolve project root by searching upward for the folder that contains `vlm`.
start_dir = Path.cwd().resolve()
project_root = next((p for p in [start_dir, *start_dir.parents] if (p / "vlm").is_dir()), start_dir)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from vlm.llava_client import query_llava_position

print(f"Project root: {project_root}")
print(f"Python executable: {sys.executable}")
print(f"CUDA available: {torch.cuda.is_available()}")
assert torch.cuda.is_available(), "CUDA is required for this notebook (expected RTX 3070)."
device = "cuda"
print(f"Using device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

Project root: D:\CodeFiles\RL-Gymnasium-PACSPL602013-Final-Project
Python executable: d:\CodeFiles\RL-Gymnasium-PACSPL602013-Final-Project\final_project_env\Scripts\python.exe
CUDA available: True
Using device: cuda
GPU: NVIDIA GeForce RTX 3070 Laptop GPU


In [2]:
# Cell 2: Reward-shaping logic and multi-threshold callback
import re
import gymnasium as gym
from stable_baselines3.common.callbacks import BaseCallback

PROMPT = "Observe the car. Is it on the LEFT slope, RIGHT slope, or BOTTOM? Reply with only one word."

def normalize_label(raw_text: str) -> str:
    cleaned = re.sub(r"[^A-Za-z]", "", raw_text).upper()
    if "LEFT" in cleaned: return "LEFT"
    if "RIGHT" in cleaned: return "RIGHT"
    return "BOTTOM"

def label_to_phi(label: str) -> float:
    if label == "LEFT": return 0.5
    if label == "RIGHT": return 1.0
    return 0.0

class VLMRewardShapingWrapper(gym.Wrapper):
    def __init__(self, env, sample_every_n: int = 128, gamma: float = 0.99):
        super().__init__(env)
        self.sample_every_n = sample_every_n
        self.gamma = gamma
        self.step_count = 0
        self.total_steps = 0
        self.current_label = "BOTTOM"
        self.current_phi = 0.0
        self.previous_phi = 0.0

    def reset(self, **kwargs):
        # Reset state to prevent episode leakage
        self.step_count = 0
        self.current_phi = 0.0
        self.previous_phi = 0.0
        self.current_label = "BOTTOM"
        return self.env.reset(**kwargs)

    def step(self, action):
        obs, reward_env, terminated, truncated, info = self.env.step(action)
        self.step_count += 1
        self.total_steps += 1
        self.previous_phi = self.current_phi

        # 1. Kinematic Reward (Dense proprioceptive feedback)
        position = float(obs[0])
        progress_reward = 0.0
        if position > -0.4:
            progress_reward = (position + 0.4) * 2.0

        # 2. Semantic Reward (Sparse VLM guidance)
        if self.step_count >= self.sample_every_n:
            self.step_count = 0
            try:
                frame = self.env.render()
                llava_reply = query_llava_position(frame=frame, model="llava:7b", prompt=PROMPT)
                self.current_label = normalize_label(llava_reply)
                phi = label_to_phi(self.current_label)
                
                # Decay VLM reliance over time
                decay = max(0.0, 1.0 - (self.total_steps / 200_000.0))
                self.current_phi = phi * decay
                
                print(f"[VLM] step={self.total_steps}, label={self.current_label}, phi={self.current_phi:.2f}")
            except Exception as e:
                self.current_label = "BOTTOM"
                self.current_phi = 0.0

        # 3. Hybrid Reward Fusion (Strict PBRS Math)
        shaping_reward = (self.gamma * self.current_phi) - self.previous_phi
        reward_total = reward_env + shaping_reward + progress_reward

        info["reward_env"] = reward_env
        info["shaping_reward"] = shaping_reward
        info["progress_reward"] = progress_reward
        info["reward_total"] = reward_total
        
        return obs, reward_total, terminated, truncated, info

class MultiThresholdCallback(BaseCallback):
    def __init__(self, thresholds=None, verbose=0):
        super().__init__(verbose)
        self.thresholds = thresholds or [-195, -190, -180, -170, -160]
        self.threshold_steps = {t: None for t in self.thresholds}

    def _on_step(self) -> bool:
        for info in self.locals.get("infos", []):
            if "episode" in info:
                episode_reward = info["episode"]["r"]
                for threshold in self.thresholds:
                    if episode_reward >= threshold and self.threshold_steps[threshold] is None:
                        self.threshold_steps[threshold] = self.num_timesteps
                        print(f"VLM-PPO reached {threshold} at step {self.num_timesteps}")
        return True

In [3]:
# Cell 3: Train VLM-PPO and export multi-threshold JSON
from pathlib import Path
import json
import sys
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
import torch

# Find project root so this notebook works from any current folder
start_dir = Path.cwd().resolve()
project_root = next((p for p in [start_dir, *start_dir.parents] if (p / "core").is_dir()), start_dir)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from core.train_manager import TrainManager

thresholds = [-195, -190, -180, -170, -160]
sample_every_n = 128
gamma = 0.99
total_timesteps_cap = 150_000

# Create a new VLM run folder
manager = TrainManager(base_dir=project_root / "logs_and_results", algo_name="vlm")
paths = manager.get_run_paths(create=True)
run_id = paths["run_id"]
run_dir = paths["run_dir"]
print(f"Start vlm run_{run_id} at {run_dir}")

# Save training parameters for this run
manager.save_params(
    run_id,
    {
        "algo": "vlm",
        "thresholds": thresholds,
        "sample_every_n": sample_every_n,
        "gamma": gamma,
        "total_timesteps": total_timesteps_cap,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
    },
)

base_env = gym.make("MountainCar-v0", render_mode="rgb_array")
monitored_env = Monitor(base_env, str(paths["monitor_csv"]))
train_env = VLMRewardShapingWrapper(monitored_env, sample_every_n=sample_every_n, gamma=gamma)

model = PPO(
    policy="MlpPolicy",
    env=train_env,
    ent_coef=0.05,
    learning_rate=0.001,
    n_steps=2048,
    verbose=1,
    tensorboard_log=str(paths["tensorboard_dir"]),
    device="cuda" if torch.cuda.is_available() else "cpu",
)

callback = MultiThresholdCallback(thresholds=thresholds)
print("Start VLM-PPO training for 150000 timesteps")
model.learn(
    total_timesteps=total_timesteps_cap,
    callback=callback,
    tb_log_name=f"ppo_vlm_run_{run_id}",
)

model_path = paths["models_dir"] / "vlm_multi_thresh"
model.save(str(model_path))
print(f"Saved model: {model_path}.zip")

vlm_results = {
    str(t): (s if s is not None else total_timesteps_cap)
    for t, s in callback.threshold_steps.items()
}
with open(paths["thresholds_json"], "w", encoding="utf-8") as f:
    json.dump(vlm_results, f, indent=2)
print(f"Saved threshold metrics: {paths['thresholds_json']}")

train_env.close()

Start vlm run_5 at D:\CodeFiles\RL-Gymnasium-PACSPL602013-Final-Project\logs_and_results\vlm\run_5
Using cuda device
Wrapping the env in a DummyVecEnv.
Start VLM-PPO training for 150000 timesteps
Logging to D:\CodeFiles\RL-Gymnasium-PACSPL602013-Final-Project\logs_and_results\vlm\run_5\tensorboard\ppo_vlm_run_5_1


d:\CodeFiles\RL-Gymnasium-PACSPL602013-Final-Project\final_project_env\Lib\site-packages\stable_baselines3\common\on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


[VLM] step=128, label=LEFT, phi=0.50
[VLM] step=328, label=LEFT, phi=0.50
[VLM] step=528, label=LEFT, phi=0.50
[VLM] step=728, label=LEFT, phi=0.50
[VLM] step=928, label=LEFT, phi=0.50
[VLM] step=1128, label=LEFT, phi=0.50
[VLM] step=1328, label=LEFT, phi=0.50
[VLM] step=1528, label=LEFT, phi=0.50
[VLM] step=1728, label=RIGHT, phi=0.99
[VLM] step=1928, label=LEFT, phi=0.50
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 200      |
|    ep_rew_mean     | -200     |
| time/              |          |
|    fps             | 57       |
|    iterations      | 1        |
|    time_elapsed    | 35       |
|    total_timesteps | 2048     |
---------------------------------
[VLM] step=2128, label=RIGHT, phi=0.99
[VLM] step=2328, label=LEFT, phi=0.49
[VLM] step=2528, label=LEFT, phi=0.49
[VLM] step=2728, label=LEFT, phi=0.49
[VLM] step=2928, label=RIGHT, phi=0.99
[VLM] step=3128, label=LEFT, phi=0.49
[VLM] step=3328, label=LEFT, phi=0.49
[VLM] step=3528,